# Task 1 - Inductive Biases and Feature Representations

Frozen ResNet-50, ViT-B/16, and OpenCLIP ViT-B-32 with controlled STL-10 interventions.

**Execution policy:** this notebook is intentionally delivered unexecuted. Set the configuration paths and switches, then run top-to-bottom when you are ready to conduct the experiment. It does not answer the report questions.

## 1. Configuration

The defaults implement the prescribed STL-10 protocol. Choose the additional color intervention and cue-conflict pairs before running, then record the hypotheses in your report.

The next cell defines all reproducibility-sensitive choices: the fixed seed, dataset location, held-out subset size, color intervention, conflict pairs, and display switches. It produces no results.

In [ ]:
# Standard library and experiment dependencies.
# Install dependencies yourself before executing: torch torchvision open_clip_torch
# scikit-learn pandas matplotlib seaborn pillow scipy tqdm
import json, random, math, copy
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageOps
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_auc_score, roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, models, transforms
from torchvision.transforms import InterpolationMode

SEED = 6304
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ROOT = Path.cwd().resolve()
if not (ROOT / "ATML-PA1.pdf").exists():
    ROOT = ROOT.parent


# Reusing one seed makes the split, subset, permutations, and head comparison reproducible.
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def save_json(value, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(value, f, indent=2)


def show_table(rows, title=None):
    frame = pd.DataFrame(rows)
    if title:
        print(title)
    display(frame)
    return frame


set_seed()

import open_clip

CONFIG = {
    "data_root": ROOT / "data",
    "download_data": False,
    "results_dir": ROOT / "task1" / "results",
    "batch_size": 64,
    "head_epochs": 50,
    "head_patience": 5,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "test_subset_per_class": 50,
    "additional_color": "hue_rotation_90",
    "cue_pairs": [
        ("airplane", "bird"),
        ("car", "truck"),
        ("cat", "dog"),
        ("deer", "horse"),
        ("monkey", "ship"),
    ],
    "adain_alpha": 0.8,
    "visual_rejection_rule": "Reject if content object is not recognizable or style texture is not visibly transferred; never use predictions.",
    "plot": True,
    "print_metrics": True,
}
RESULTS = Path(CONFIG["results_dir"])
RESULTS.mkdir(parents=True, exist_ok=True)
CLASSES = ["airplane", "bird", "car", "cat", "deer", "dog", "horse", "monkey", "ship", "truck"]


## 2. Data split and shared 224x224 interventions

All backbones receive identical RGB images before their model-specific normalization. The saved manifest makes the class-balanced test subset recoverable.

The next cell creates the fixed stratified 80/20 train/validation split and 500-image balanced test manifest, then defines shared pixel-space interventions. It saves `test_subset_manifest.json`.

In [ ]:
raw_train = datasets.STL10(CONFIG["data_root"], split="train", download=CONFIG["download_data"])
raw_test = datasets.STL10(CONFIG["data_root"], split="test", download=CONFIG["download_data"])
labels = np.asarray(raw_train.labels)
rng = np.random.default_rng(SEED)
train_indices, val_indices = [], []
for label in range(10):
    indices = np.flatnonzero(labels == label)
    rng.shuffle(indices)
    cut = int(0.8 * len(indices))
    train_indices.extend(indices[:cut])
    val_indices.extend(indices[cut:])
test_labels = np.asarray(raw_test.labels)
chosen_test = []
for label in range(10):
    candidates = np.flatnonzero(test_labels == label)
    rng.shuffle(candidates)
    chosen_test.extend(candidates[: CONFIG["test_subset_per_class"]])
save_json(
    {"seed": SEED, "indices": list(map(int, chosen_test))}, RESULTS / "test_subset_manifest.json"
)


def common_image(image):
    return transforms.CenterCrop(224)(
        transforms.Resize(256, InterpolationMode.BICUBIC)(image.convert("RGB"))
    )


def grayscale(image):
    return ImageOps.grayscale(image).convert("RGB")


def hue_rotation_90(image):
    hsv = np.asarray(image.convert("HSV")).copy()
    hsv[..., 0] = (hsv[..., 0].astype(np.uint16) + 64) % 256
    return Image.fromarray(hsv, "HSV").convert("RGB")


def translate_reflect(image, pixels, direction):
    array = np.asarray(image)
    pad = pixels
    padded = np.pad(array, ((pad, pad), (pad, pad), (0, 0)), mode="reflect")
    dx, dy = {"left": (-pixels, 0), "right": (pixels, 0), "up": (0, -pixels), "down": (0, pixels)}[
        direction
    ]
    y, x = pad + dy, pad + dx
    return Image.fromarray(padded[y : y + array.shape[0], x : x + array.shape[1]])


# The permutation is seeded from the image identifier so every backbone sees the same shuffled image.
def shuffle_4x4(image, index):
    array = np.asarray(image)
    h, w = array.shape[:2]
    rng = np.random.default_rng(SEED + int(index))
    order = rng.permutation(16)
    order = order if not np.array_equal(order, np.arange(16)) else np.roll(order, 1)
    pieces = [
        [array[i * h // 4 : (i + 1) * h // 4, j * w // 4 : (j + 1) * w // 4] for j in range(4)]
        for i in range(4)
    ]
    out = np.empty_like(array)
    for target, source in enumerate(order):
        ti, tj = divmod(target, 4)
        si, sj = divmod(source, 4)
        out[ti * h // 4 : (ti + 1) * h // 4, tj * w // 4 : (tj + 1) * w // 4] = pieces[si][sj]
    return Image.fromarray(out)


## 3. Frozen representations and linear heads

The next cell loads the three required pretrained backbones, freezes them, extracts final representations, and trains one early-stopped linear head per backbone. Its outputs are trained heads and feature tensors.

In [ ]:
resnet_w = models.ResNet50_Weights.IMAGENET1K_V2
vit_w = models.ViT_B_16_Weights.IMAGENET1K_V1
resnet = models.resnet50(weights=resnet_w)
resnet.fc = nn.Identity()
vit = models.vit_b_16(weights=vit_w)
vit.heads = nn.Identity()
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="openai"
)
for model in (resnet, vit, clip_model):
    model.to(DEVICE).eval()
    for parameter in model.parameters():
        parameter.requires_grad = False
normalizers = {"resnet": resnet_w.transforms(), "vit": vit_w.transforms(), "clip": clip_preprocess}
backbones = {"resnet": resnet, "vit": vit, "clip": clip_model}


# Each model receives the same PIL image; only this step applies its required normalization.
def feature(model_name, image_batch):
    # image_batch is a list of common PIL images; normalization happens only here.
    tensor = torch.stack([normalizers[model_name](im) for im in image_batch]).to(DEVICE)
    with torch.inference_mode():
        output = (
            clip_model.encode_image(tensor)
            if model_name == "clip"
            else backbones[model_name](tensor)
        )
        return F.normalize(output.float(), dim=1).cpu()


def extract(model_name, dataset, indices, transform=lambda im, image_id: im):
    features, ys = [], []
    for start in range(0, len(indices), CONFIG["batch_size"]):
        batch_ids = indices[start : start + CONFIG["batch_size"]]
        samples = [dataset[int(i)] for i in batch_ids]
        images = [
            transform(common_image(sample[0]), int(image_id))
            for sample, image_id in zip(samples, batch_ids)
        ]
        features.append(feature(model_name, images))
        ys.extend(int(sample[1]) for sample in samples)
    return torch.cat(features), torch.tensor(ys)


train_features = {name: extract(name, raw_train, train_indices) for name in backbones}
val_features = {name: extract(name, raw_train, val_indices) for name in backbones}
heads = {}
for name, (x_train, y_train) in train_features.items():
    head = nn.Linear(x_train.shape[1], 10).to(DEVICE)
    optimizer = torch.optim.AdamW(
        head.parameters(), lr=CONFIG["learning_rate"], weight_decay=CONFIG["weight_decay"]
    )
    best, stale, best_state = -1, 0, None
    for epoch in range(CONFIG["head_epochs"]):
        head.train()
        optimizer.zero_grad()
        loss = F.cross_entropy(head(x_train.to(DEVICE)), y_train.to(DEVICE))
        loss.backward()
        optimizer.step()
        head.eval()
        score = accuracy_score(
            val_features[name][1], head(val_features[name][0].to(DEVICE)).argmax(1).cpu()
        )
        if score > best:
            best, stale, best_state = score, 0, copy.deepcopy(head.state_dict())
        else:
            stale += 1
        if stale >= CONFIG["head_patience"]:
            break
    head.load_state_dict(best_state)
    heads[name] = head.eval()


## 4. Clean, color, and patch evaluation

Set `print_metrics` or `plot` in the configuration to control notebook output.

The next cell evaluates clean, grayscale, hue-rotation, and patch-shuffle inputs. It produces top-1 accuracy, macro-F1, mean maximum confidence, accuracy change, and prediction consistency.

In [ ]:
tokenizer = open_clip.get_tokenizer("ViT-B-32")
with torch.inference_mode():
    text = F.normalize(
        clip_model.encode_text(
            tokenizer([f"a photo of a {name}." for name in CLASSES]).to(DEVICE)
        ).float(),
        dim=1,
    )


def predict(name, feats):
    if name == "clip_zero_shot":
        return (100 * feats.to(DEVICE) @ text.T).softmax(1).cpu()
    return heads[name](feats.to(DEVICE)).softmax(1).cpu()


def evaluate_condition(name, transform):
    feats, y = extract(
        "clip" if name == "clip_zero_shot" else name, raw_test, chosen_test, transform
    )
    probs = predict(name, feats)
    pred = probs.argmax(1)
    return {
        "accuracy": accuracy_score(y, pred),
        "macro_f1": f1_score(y, pred, average="macro"),
        "mean_max_confidence": probs.max(1).values.mean().item(),
        "pred": pred,
        "features": feats,
        "labels": y,
    }


conditions = {
    "clean": lambda image, image_id: image,
    "grayscale": lambda image, image_id: grayscale(image),
    "hue_rotation_90": lambda image, image_id: hue_rotation_90(image),
    "patch_shuffle": None,
}
all_results = {}
for model_name in ["resnet", "vit", "clip", "clip_zero_shot"]:
    clean = evaluate_condition(model_name, conditions["clean"])
    all_results[(model_name, "clean")] = clean
    for condition in ["grayscale", "hue_rotation_90"]:
        result = evaluate_condition(model_name, conditions[condition])
        result["consistency"] = (result["pred"] == clean["pred"]).float().mean().item()
        all_results[(model_name, condition)] = result
    # Each test image uses a distinct but deterministic non-identity permutation.
    shuffled = extract(
        "clip" if model_name == "clip_zero_shot" else model_name,
        raw_test,
        chosen_test,
        lambda image, image_id: shuffle_4x4(image, image_id),
    )
    y = shuffled[1]
    probs = predict(model_name, shuffled[0])
    pred = probs.argmax(1)
    all_results[(model_name, "patch_shuffle")] = {
        "accuracy": accuracy_score(y, pred),
        "macro_f1": f1_score(y, pred, average="macro"),
        "mean_max_confidence": probs.max(1).values.mean().item(),
        "pred": pred,
        "features": shuffled[0],
        "labels": y,
        "consistency": (pred == clean["pred"]).float().mean().item(),
    }
rows = []
for (model, condition), value in all_results.items():
    row = {
        "model": model,
        "condition": condition,
        **{k: round(value[k], 4) for k in ["accuracy", "macro_f1", "mean_max_confidence"]},
    }
    if condition != "clean":
        row["accuracy_change"] = round(
            value["accuracy"] - all_results[(model, "clean")]["accuracy"], 4
        )
        row["consistency"] = round(value["consistency"], 4)
    rows.append(row)
if CONFIG["print_metrics"]:
    show_table(rows, "Clean and intervention metrics")
save_json(rows, RESULTS / "condition_metrics.json")


## 5. Translation curve

The next cell evaluates the four cardinal translations at 0, 8, 16, and 32 pixels. It saves a metric table and plots accuracy and prediction consistency against displacement.

In [ ]:
translation_rows = []
for model_name in ["resnet", "vit", "clip", "clip_zero_shot"]:
    clean_pred = all_results[(model_name, "clean")]["pred"]
    for pixels in [0, 8, 16, 32]:
        direction_predictions = []
        for direction in ["left", "right", "up", "down"]:
            transformed = evaluate_condition(
                model_name, lambda im, image_id, p=pixels, d=direction: translate_reflect(im, p, d)
            )
            direction_predictions.append(transformed["pred"])
        stacked = torch.stack(direction_predictions)
        y = all_results[(model_name, "clean")]["labels"]
        translation_rows.append(
            {
                "model": model_name,
                "pixels": pixels,
                "accuracy": np.mean([accuracy_score(y, p) for p in direction_predictions]),
                "consistency": (stacked == clean_pred).float().mean().item(),
            }
        )
translation_frame = pd.DataFrame(translation_rows)
save_json(translation_rows, RESULTS / "translation_metrics.json")
if CONFIG["plot"]:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for model, frame in translation_frame.groupby("model"):
        axes[0].plot(frame.pixels, frame.accuracy, "o-", label=model)
        axes[1].plot(frame.pixels, frame.consistency, "o-", label=model)
    axes[0].set(title="Translation accuracy", xlabel="displacement (pixels)", ylabel="accuracy")
    axes[1].set(
        title="Prediction consistency", xlabel="displacement (pixels)", ylabel="consistency"
    )
    axes[1].legend()
    plt.show()


## 6. Cue conflicts (AdaIN)

Install the cited AdaIN implementation and point `ADAIN_DIR` to it before executing this section. Generate conflicts, inspect them using the rejection rule configured above, record accepted/rejected IDs, and only then evaluate.

The following setup creates an auditable conflict manifest before any prediction. After manual visual acceptance, it computes shape, texture, other, shape-bias, and coverage counts.

In [ ]:
ADAIN_DIR = (
    ROOT / "external" / "pytorch-AdaIN"
)  # cite this dependency in README; do not copy its source into this repository
CONFLICT_MANIFEST = RESULTS / "cue_conflict_manifest.json"


# This explicit hook keeps stylization generation separate from prediction. Implement `adain_stylize` using the cited repo's
# `net.py`, `function.py`, decoder weights, and VGG-normalised weights after installing that dependency.
def adain_stylize(content_image, style_image, alpha=CONFIG["adain_alpha"]):
    raise NotImplementedError(
        "Configure the cited AdaIN dependency before generating cue conflicts."
    )


def build_conflict_candidates():
    candidates = []
    for shape_name, texture_name in CONFIG["cue_pairs"]:
        a, b = CLASSES.index(shape_name), CLASSES.index(texture_name)
        for shape_label, style_label in [(a, b), (b, a)]:
            content_ids = [i for i in chosen_test if raw_test.labels[i] == shape_label]
            style_ids = [i for i in chosen_test if raw_test.labels[i] == style_label]
            for content_id, style_id in zip(content_ids, style_ids):
                candidates.append(
                    {
                        "content_id": int(content_id),
                        "style_id": int(style_id),
                        "shape_label": shape_label,
                        "texture_label": style_label,
                        "accepted": None,
                        "rejection_reason": None,
                    }
                )
    return candidates


# Save candidate records, manually set accepted/rejection_reason after visual inspection, then reload the manifest.
# Accepted conflicts must total at least 200 and be balanced across pairs/directions as closely as possible.
if not CONFLICT_MANIFEST.exists():
    save_json(build_conflict_candidates(), CONFLICT_MANIFEST)


# Shape bias excludes "other" predictions; coverage reports how much of the sample was eligible.
def cue_summary(predictions, records):
    shape = sum(int(p) == r["shape_label"] for p, r in zip(predictions, records))
    texture = sum(int(p) == r["texture_label"] for p, r in zip(predictions, records))
    total = len(records)
    return {
        "shape": shape,
        "texture": texture,
        "other": total - shape - texture,
        "shape_bias_percent": 100 * shape / max(shape + texture, 1),
        "coverage_percent": 100 * (shape + texture) / max(total, 1),
    }


## 7. Representation stability and projections

The next cell compares paired clean/transformed backbone features. It produces cosine-stability rows and one combined-condition t-SNE plot per backbone/intervention.

In [ ]:
# After cue conflicts are accepted, add their paired features to `paired_conditions`. This block handles all required conditions.
paired_conditions = {
    "grayscale": "grayscale",
    "patch_shuffle": "patch_shuffle",
}  # translation may use a chosen displacement, e.g. 32 pixels
stability_rows = []
for model_name in ["resnet", "vit", "clip"]:
    clean_features = all_results[(model_name, "clean")]["features"]
    for label, condition in paired_conditions.items():
        transformed = all_results[(model_name, condition)]["features"]
        cosine = F.cosine_similarity(clean_features, transformed).mean().item()
        stability_rows.append(
            {"backbone": model_name, "intervention": label, "cosine_stability": cosine}
        )
        combined = torch.cat([clean_features, transformed]).numpy()
        projection = TSNE(
            n_components=2, random_state=SEED, init="pca", learning_rate="auto", perplexity=30
        ).fit_transform(combined)
        labels = np.tile(all_results[(model_name, "clean")]["labels"].numpy(), 2)
        markers = np.array(["clean"] * len(clean_features) + [label] * len(transformed))
        if CONFIG["plot"]:
            plot = pd.DataFrame(
                {
                    "x": projection[:, 0],
                    "y": projection[:, 1],
                    "class": labels,
                    "condition": markers,
                }
            )
            sns.scatterplot(
                data=plot, x="x", y="y", hue="class", style="condition", palette="tab10", s=25
            )
            plt.title(f"{model_name}: clean vs {label}")
            plt.show()
if CONFIG["print_metrics"]:
    show_table(stability_rows, "Representation cosine stability")
save_json(stability_rows, RESULTS / "representation_stability.json")
